# LLM screening and hold-out validation

This notebook applies six fixed eligibility questions to the review corpus and evaluates the resulting decisions against a manually labeled hold-out.

If the hold-out file is absent, the notebook creates a 50-record review sheet and stops. Complete its `label` values (`true`/`false`) manually, save it, and rerun. If corpus-level LLM scores already exist, they are reused; no API call or corpus rescreening is performed. The evaluation reports hold-out metrics and saves them separately from corpus outputs.

In [14]:
import json
import os
import re
import time
import math
import hashlib
import unicodedata
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix
from scipy.stats import beta as beta_dist

pd.set_option("display.max_colwidth", 120)

## Configuration

Paths, screening settings, and output locations.

In [15]:
CONFIG = {
    "MAIN_DATA_PATH": "../data/03_document_search_results_with_intros_and_contributions.json",
    "EXCLUDED_DOCUMENT_TYPES": ["Note", "Retracted"],
    "HOLDOUT_PATH": "../model/dataset/screening_holdout.json",
    "HOLDOUT_SIZE": 100,
    "HOLDOUT_SEED": 42,
    "HOLDOUT_DEV_SIZE": 50,

    # Reuse this scored corpus for hold-out validation when it is present.
    "EXISTING_CORPUS_SCORES_PATH": "../data/04_llm_scores_corpus_medium.json",
    "RUN_ON": "corpus",
    "EVALUATE_TEST": True,

    "LLM_TEXT_FIELDS": ["abstract", "contribution"],
    "STUDENT_TEXT_FIELDS": ["contribution", "abstract"],

    "MODELS": [
        {"name": "gpt-6-luna", "model": "gpt-6-luna", "reasoning_effort": "medium",
         "price_in": 0.10, "price_out": 0.50},
    ],
    "MAX_RETRIES": 4,

    "EMBEDDING_MODEL_NAME": "sentence-transformers/all-mpnet-base-v2",
    "ALPHA": 0.5,
    "BETA": 0.5,

    "PSEUDO_LABEL_FRACTIONS": [0.15, 0.25, 0.35],
    "PSEUDO_LABEL_DIR": "../model/dataset",
    "OUTPUT_DIR": "../data",
    "CACHE_DIR": "../data/llm_cache",
}
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
os.makedirs(CONFIG["CACHE_DIR"], exist_ok=True)

## Screening criteria

The six questions use `yes` = 1, `unclear` = 0.5, and `no` = 0. The fixed inclusion rule is defined below.

In [16]:
QUESTIONS = {
    "Q1": "Are the images or videos processed with computer vision or other automated image analysis "
          "(e.g., detection, tracking, segmentation, automated trajectory extraction)? Answer \"yes\" only if the "
          "abstract states or clearly implies automated analysis; answer \"no\" if the footage is only watched, "
          "coded or annotated manually, and \"unclear\" if the abstract does not say how the footage was analysed.",
    "Q2": "Is the study primarily concerned with the safety, behaviour or movement of pedestrians, cyclists "
          "(including e-bikes) or scooter users (including e-scooters)? Answer \"no\" if the focus is on drivers, "
          "motorcyclists, autonomous or automated vehicles, driver-assistance systems, robots or personal mobility "
          "vehicles, or if the main purpose is to warn drivers or support driving using a vehicle-mounted camera "
          "(e.g., a dash-cam), even if pedestrians or cyclists appear in the data.",
    "Q3": "Does the study use the visual data to draw conclusions about a specific real-world space used for "
          "walking, cycling or scooter riding (e.g., a street, crossing, intersection, lane, sidewalk or station), "
          "either its physical environment or how people move through it (e.g., counts, flows, speeds, crossing "
          "behaviour, conflicts), or to detect obstacles and hazards along the route of a pedestrian or rider "
          "(e.g., assistive navigation)? Answer \"no\" if the contribution is only an algorithm evaluated on "
          "benchmark datasets, a simulation, a structural or engineering test, positioning or signal-reception "
          "research (e.g., GNSS), or security, crime or public-health surveillance (e.g., anomaly detection, "
          "social distancing).",
    "Q4": "Were the images or videos recorded in real-world locations by cameras deployed there (fixed, "
          "bicycle- or rider-mounted, wearable, or drone-based), including existing CCTV or traffic cameras, field "
          "video and public datasets recorded this way, rather than taken from existing image collections such as "
          "Google Street View, satellite imagery or social media photos? Answer \"yes\" if the abstract mentions "
          "field, CCTV, traffic-camera, drone or on-site video or footage from real locations, even without "
          "describing the camera setup; answer \"unclear\" only if it gives no indication of where the visual "
          "data come from.",
    "Q5": "Do the visual data include conventional visible-light camera images or video? Answer \"yes\" unless the "
          "abstract states that only thermal/infrared cameras, LiDAR/laser scanning or non-camera sensors were "
          "used; the abstract does not need to mention RGB explicitly.",
    "Q6": "Are the images or videos analysed as data about the space or its users, rather than only shown to "
          "participants as stimuli (e.g., in virtual reality, video-based questionnaires, eye-tracking or "
          "crossing-decision experiments)?",
}

# Used for criteria-level re-ranking: the scope as one paragraph.
CRITERIA_PARAGRAPH = (
    "Studies that monitor soft mobility (pedestrians, cyclists and scooter users) with cameras deployed in the "
    "real environment (fixed, mobile, wearable or drone cameras, including CCTV and traffic cameras), using "
    "computer vision on images or video to assess the physical environment of specific streets, crossings, "
    "lanes or stations, or how people move through them."
)

SYSTEM_PROMPT = """You are a researcher screening titles and abstracts of scientific papers for a survey on \
how visual sensing technologies and computer vision methods are used for monitoring soft mobility \
(pedestrians, cyclists including e-bikes, and scooter users including e-scooters).

Answer each question about the paper using only the title and abstract. Base each answer on what the \
abstract actually states, not on what is typical for the topic. For each question answer:
- "yes" if the abstract supports that the criterion is met,
- "no" if the abstract indicates it is not met,
- "unclear" if the abstract does not give enough information to decide.
Follow any question-specific instruction about when to answer "yes", "no" or "unclear".

Questions:
""" + "\n".join(f"{k}. {v}" for k, v in QUESTIONS.items()) + """

Respond with ONLY a JSON object, no other text, in exactly this form:
{"Q1": {"answer": "yes|no|unclear", "evidence": "<one short sentence citing the abstract>"}, \
"Q2": {...}, "Q3": {...}, "Q4": {...}, "Q5": {...}, "Q6": {...}}"""

ANSWER_SCORE = {"yes": 1.0, "unclear": 0.5, "no": 0.0}
PROMPT_VERSION = hashlib.sha1(SYSTEM_PROMPT.encode("utf-8")).hexdigest()[:10]
print(f"Prompt version: {PROMPT_VERSION}  (~{len(SYSTEM_PROMPT) // 4} tokens)")


def build_user_message(title, text):
    return f"Title: {title}\n\nAbstract: {text}"

Prompt version: 908809cd0c  (~918 tokens)


## Helper functions

In [17]:
def first_text(record, fields):
    for field in fields:
        val = record.get(field)
        if isinstance(val, str) and val.strip():
            return val.strip()
    return None


def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_doi(doi):
    if not isinstance(doi, str) or not doi.strip():
        return None
    doi = doi.strip().lower()
    doi = re.sub(r"^(https?://)?(dx\.)?doi\.org/", "", doi)
    doi = re.sub(r"^doi:\s*", "", doi)
    return doi or None


def record_key(record):
    """DOI, else Scopus id, else normalized title."""
    doi = normalize_doi(record.get("doi"))
    if doi:
        return f"doi:{doi}"
    scopus_id = record.get("scopus_id")
    if scopus_id is not None and str(scopus_id).strip():
        return f"scopus:{str(scopus_id).strip()}"
    return f"title:{normalize_text(record.get('title'))}"


def to_label(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, np.integer)) and value in (0, 1):
        return int(value)
    if isinstance(value, str) and value.strip().lower() in ("true", "1", "false", "0"):
        return int(value.strip().lower() in ("true", "1"))
    if isinstance(value, str) and not value.strip():
        return None
    raise ValueError(f"Unexpected label value: {value!r}")


def read_json(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def summarize_predictions(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel())
    n = len(y_true)
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    f1 = 2 * precision * recall / (precision + recall) if (tp and precision + recall) else 0.0
    return {"selected": tp + fp, "n": n, "recall": recall, "precision": precision, "f1": f1,
            "excluded": (tn + fn) / n if n else np.nan, "tn": tn, "fp": fp, "fn": fn, "tp": tp}


def threshold_for_max_f1(y_true, scores):
    """Threshold maximizing F1; ties go to the lower threshold (higher recall)."""
    scores = np.asarray(scores, dtype=float)
    best_t, best_f1 = None, -1.0
    for t in np.unique(scores):  # ascending
        f1 = summarize_predictions(y_true, scores >= t)["f1"]
        if f1 > best_f1:
            best_t, best_f1 = float(t), f1
    return best_t


def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    lo = beta_dist.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0.0
    hi = beta_dist.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1.0
    return (float(lo), float(hi))

## Step 1 — Corpus and hold-out

In [18]:
df_corpus = pd.DataFrame(read_json(CONFIG["MAIN_DATA_PATH"]))
if "document_type" in df_corpus:
    dropped = df_corpus["document_type"].isin(CONFIG["EXCLUDED_DOCUMENT_TYPES"])
    print(f"Removed {int(dropped.sum())} excluded document types")
    df_corpus = df_corpus[~dropped].reset_index(drop=True)

df_corpus["record_key"] = df_corpus.apply(lambda r: record_key(r.to_dict()), axis=1)
df_corpus = df_corpus.drop_duplicates("record_key").reset_index(drop=True)
df_corpus["llm_text"] = df_corpus.apply(lambda r: first_text(r.to_dict(), CONFIG["LLM_TEXT_FIELDS"]), axis=1)
df_corpus["student_text"] = df_corpus.apply(lambda r: first_text(r.to_dict(), CONFIG["STUDENT_TEXT_FIELDS"]), axis=1)
print(f"Corpus: {len(df_corpus)} records; {int(df_corpus['llm_text'].isna().sum())} without usable text")

if not os.path.exists(CONFIG["HOLDOUT_PATH"]):
    candidates = df_corpus[df_corpus["llm_text"].notna()].copy()
    sample_size = min(CONFIG["HOLDOUT_SIZE"], len(candidates))
    if sample_size == 0:
        raise ValueError("Cannot create a hold-out: no corpus records have usable text.")
    sample = candidates.sample(n=sample_size, random_state=CONFIG["HOLDOUT_SEED"])
    review_rows = []
    for _, row in sample.iterrows():
        record = row.to_dict()
        selected_field = next(
            (field for field in CONFIG["STUDENT_TEXT_FIELDS"]
             if isinstance(record.get(field), str) and record[field].strip()),
            "abstract",
        )
        review_rows.append({
            "record_key": record["record_key"],
            "doi": record.get("doi") if pd.notna(record.get("doi")) else None,
            "title": record.get("title") if pd.notna(record.get("title")) else None,
            "text_field": selected_field,
            "text": first_text(record, CONFIG["STUDENT_TEXT_FIELDS"]),
            "label": None,
            "notes": "",
        })
    os.makedirs(os.path.dirname(CONFIG["HOLDOUT_PATH"]), exist_ok=True)
    with open(CONFIG["HOLDOUT_PATH"], "w", encoding="utf-8") as f:
        json.dump(review_rows, f, indent=2, ensure_ascii=False)
    print(f"Created {CONFIG['HOLDOUT_PATH']} with {sample_size} records for manual labeling.")
    raise RuntimeError(
        "Manual review required: fill every holdout label with true/false, save the file, "
        "then rerun the notebook. No screening or evaluation was run."
    )

# Load the manually labeled hold-out and preserve its row order for the dev/test split.
df_hold = pd.DataFrame(read_json(CONFIG["HOLDOUT_PATH"]))
if "record_key" not in df_hold:
    df_hold["record_key"] = df_hold.apply(lambda r: record_key(r.to_dict()), axis=1)
if "label" not in df_hold:
    raise ValueError("The hold-out file must contain a 'label' column with true/false values.")
df_hold["label"] = df_hold["label"].apply(to_label)
if df_hold["label"].isna().any():
    raise RuntimeError("Manual review incomplete: label every hold-out record true/false, then rerun.")
df_hold["label"] = df_hold["label"].astype(int)
df_hold["split"] = np.where(np.arange(len(df_hold)) < CONFIG["HOLDOUT_DEV_SIZE"], "dev", "test")

missing = ~df_hold["record_key"].isin(df_corpus["record_key"])
if missing.any():
    raise RuntimeError(
        f"Hold-out/corpus mismatch: {int(missing.sum())}/{len(df_hold)} labeled records are absent "
        "from the configured corpus. Refusing to report partial validation; use matching inputs "
        "or create and label a hold-out from this corpus."
    )
print(df_hold.groupby("split")["label"].agg(["count", "sum"]).rename(columns={"sum": "in_scope"}))

holdout_keys = set(df_hold["record_key"])
if CONFIG["RUN_ON"] == "holdout":
    df_run = df_corpus[df_corpus["record_key"].isin(holdout_keys)].copy()
else:
    df_run = df_corpus.copy()
df_run = df_run[df_run["llm_text"].notna()].reset_index(drop=True)

USE_EXISTING_CORPUS_SCORES = (
    CONFIG["RUN_ON"] == "corpus"
    and os.path.exists(CONFIG["EXISTING_CORPUS_SCORES_PATH"])
)
if USE_EXISTING_CORPUS_SCORES:
    df_existing = pd.DataFrame(read_json(CONFIG["EXISTING_CORPUS_SCORES_PATH"]))
    required = {"record_key", "veto_pass", "llm_mean", "llm_rerank", "answers"}
    missing_columns = required - set(df_existing.columns)
    if missing_columns:
        raise ValueError(f"Existing corpus scores lack required columns: {sorted(missing_columns)}")
    unscored_holdout = holdout_keys - set(df_existing["record_key"])
    if unscored_holdout:
        raise ValueError(f"Existing scores are missing {len(unscored_holdout)} hold-out records.")
    print(f"Reusing existing corpus scores: {CONFIG['EXISTING_CORPUS_SCORES_PATH']}")
else:
    print(f"Records to screen ({CONFIG['RUN_ON']}): {len(df_run)}")

Removed 0 excluded document types
Corpus: 694 records; 0 without usable text
       count  in_scope
split                 
dev       50        25
test      50        20
Reusing existing corpus scores: ../data/04_llm_scores_corpus_medium.json


## Step 2 — Cost estimate

Shown only when new LLM requests are required.

In [19]:
if USE_EXISTING_CORPUS_SCORES:
    print("Existing corpus scores found; no new LLM requests are needed.")
else:
    approx_in = sum((len(SYSTEM_PROMPT) + len(build_user_message(t, x))) / 4
                    for t, x in zip(df_run["title"].fillna(""), df_run["llm_text"]))
    approx_out = len(df_run) * 60 * len(QUESTIONS)
    total = 0.0
    for m in CONFIG["MODELS"]:
        cost = approx_in / 1e6 * m["price_in"] + approx_out / 1e6 * m["price_out"]
        total += cost
        print(f"{m['name']:>22s}: ~{approx_in / 1e6:.2f}M in, ~{approx_out / 1e6:.2f}M out -> ~US$ {cost:.2f}")
    print(f"{'total':>22s}: ~US$ {total:.2f}")

Existing corpus scores found; no new LLM requests are needed.


## Step 3 — LLM client

The API client is initialized only when the scored corpus is not available for reuse.

In [20]:
if not USE_EXISTING_CORPUS_SCORES:
    import getpass
    from openai import OpenAI

    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (hidden input): ").strip()
    client = OpenAI()
    _UNSUPPORTED = set()

    def call_openai(m, user):
        base = dict(
            model=m["model"],
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user},
            ],
            response_format={"type": "json_object"},
        )
        optional = {"temperature": 0}
        if m.get("reasoning_effort"):
            optional["reasoning_effort"] = m["reasoning_effort"]
        while True:
            extra = {k: v for k, v in optional.items() if (m["model"], k) not in _UNSUPPORTED}
            try:
                response = client.chat.completions.create(**base, **extra)
                return (response.choices[0].message.content,
                        response.usage.prompt_tokens, response.usage.completion_tokens)
            except Exception as exc:
                rejected = [key for key in extra if key in str(exc)]
                if not rejected:
                    raise
                _UNSUPPORTED.update((m["model"], key) for key in rejected)
                print(f"{m['model']}: unsupported parameter(s) {rejected}; retrying without them")

    try:
        client.models.retrieve(CONFIG["MODELS"][0]["model"])
        print("API key and model are available.")
    except Exception as exc:
        raise RuntimeError(f"OpenAI check failed (key, credits or model ID): {exc}") from exc

    def parse_answers(raw):
        match = re.search(r"\{.*\}", raw or "", re.DOTALL)
        if not match:
            return None
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            return None
        out = {}
        for question in QUESTIONS:
            item = data.get(question)
            if isinstance(item, str):
                item = {"answer": item}
            if not isinstance(item, dict):
                return None
            answer = str(item.get("answer", "")).strip().lower()
            if answer not in ANSWER_SCORE:
                return None
            out[question] = (answer, str(item.get("evidence", "")))
        return out

    def _cache_path(model_config):
        effort = model_config.get("reasoning_effort") or "default"
        return os.path.join(
            CONFIG["CACHE_DIR"],
            f"{model_config['name']}__{effort}__{PROMPT_VERSION}.jsonl",
        )

    def load_cache(model_config):
        cache = {}
        path = _cache_path(model_config)
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    row = json.loads(line)
                    cache[row["record_key"]] = row
        return cache

    def screen_record(model_config, key, title, text):
        user = build_user_message(title or "", text)
        last_error = None
        for attempt in range(CONFIG["MAX_RETRIES"]):
            try:
                raw, tokens_in, tokens_out = call_openai(model_config, user)
                answers = parse_answers(raw)
                if answers is not None:
                    return {
                        "record_key": key,
                        "model": model_config["name"],
                        "model_id": model_config["model"],
                        "prompt_version": PROMPT_VERSION,
                        "tokens_in": tokens_in,
                        "tokens_out": tokens_out,
                        "answers": {q: answer for q, (answer, _) in answers.items()},
                        "evidence": {q: evidence for q, (_, evidence) in answers.items()},
                        "raw": raw,
                    }
                last_error = f"unparseable output: {raw[:200]!r}"
            except Exception as exc:
                last_error = repr(exc)
                if "insufficient_quota" in str(exc) or "invalid_api_key" in str(exc):
                    raise RuntimeError(f"Stopping: {exc}") from exc
            time.sleep(2 ** attempt)
        return {
            "record_key": key,
            "model": model_config["name"],
            "model_id": model_config["model"],
            "prompt_version": PROMPT_VERSION,
            "tokens_in": 0,
            "tokens_out": 0,
            "answers": None,
            "evidence": None,
            "error": last_error,
        }

## Step 4 — Run screening when needed

In [21]:
results = {}
if USE_EXISTING_CORPUS_SCORES:
    print("Skipped LLM screening; using the existing scored corpus for hold-out evaluation.")
else:
    for model_config in CONFIG["MODELS"]:
        cache = load_cache(model_config)
        todo = df_run[~df_run["record_key"].isin(
            [key for key, result in cache.items() if result.get("answers")]
        )]
        print(f"{model_config['name']}: {len(df_run) - len(todo)} cached, {len(todo)} to query")
        with open(_cache_path(model_config), "a", encoding="utf-8") as f:
            for _, row in tqdm(todo.iterrows(), total=len(todo), desc=model_config["name"]):
                result = screen_record(
                    model_config, row["record_key"], row.get("title"), row["llm_text"]
                )
                cache[row["record_key"]] = result
                if result.get("answers"):
                    f.write(json.dumps(result, ensure_ascii=False) + "\n")
        results[model_config["name"]] = cache
        failed = [key for key in df_run["record_key"] if not (cache.get(key) or {}).get("answers")]
        print(f"  failed: {len(failed)}")

Skipped LLM screening; using the existing scored corpus for hold-out evaluation.


## Step 5 — Decisions and ranking scores

A record passes the fixed rule when no question is `no` and no more than one is `unclear`. The ranking scores are retained for comparison.

In [22]:
MODEL_NAMES = [model["name"] for model in CONFIG["MODELS"]]
QS = list(QUESTIONS)

if USE_EXISTING_CORPUS_SCORES:
    score_columns = [
        "record_key", "n_no", "n_unclear", "n_missing_q", "veto_pass",
        "llm_mean", "llm_rerank", "answers",
    ]
    df_scores = df_existing[score_columns].copy()
    for question in QS:
        df_scores[f"vote__{question}"] = df_scores["answers"].map(
            lambda answers: (answers.get(question) or {}).get("score")
            if isinstance(answers, dict) else np.nan
        )
    print(f"Loaded {len(df_scores)} existing scored records.")
else:
    rows = []
    for _, record in df_run.iterrows():
        row = {"record_key": record["record_key"]}
        for name in MODEL_NAMES:
            result = results[name].get(record["record_key"]) or {}
            for question in QS:
                answer = (result.get("answers") or {}).get(question)
                row[f"{name}__{question}"] = ANSWER_SCORE.get(answer, np.nan)
        rows.append(row)
    df_scores = pd.DataFrame(rows)

    for question in QS:
        df_scores[f"vote__{question}"] = df_scores[
            [f"{name}__{question}" for name in MODEL_NAMES]
        ].mean(axis=1)
    vote_cols = [f"vote__{question}" for question in QS]
    df_scores["n_no"] = (df_scores[vote_cols] == 0.0).sum(axis=1)
    df_scores["n_unclear"] = (
        (df_scores[vote_cols] > 0.0) & (df_scores[vote_cols] < 1.0)
    ).sum(axis=1)
    df_scores["n_missing_q"] = df_scores[vote_cols].isna().sum(axis=1)
    df_scores["veto_pass"] = (
        (df_scores["n_no"] == 0)
        & (df_scores["n_unclear"] <= 1)
        & (df_scores["n_missing_q"] == 0)
    )
    df_scores["llm_mean"] = df_scores[vote_cols].mean(axis=1)

    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer(CONFIG["EMBEDDING_MODEL_NAME"])
    texts = (df_run["title"].fillna("") + ". " + df_run["llm_text"]).tolist()
    emb_docs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=len(texts) > 64)
    emb_q = embedder.encode(list(QUESTIONS.values()), normalize_embeddings=True)
    emb_crit = embedder.encode([CRITERIA_PARAGRAPH], normalize_embeddings=True)[0]
    sim_q = emb_docs @ emb_q.T
    sim_crit = emb_docs @ emb_crit
    alpha, beta = CONFIG["ALPHA"], CONFIG["BETA"]
    per_q = (1 - alpha) * df_scores[vote_cols].to_numpy() + alpha * sim_q
    df_scores["llm_rerank"] = (1 - beta) * np.nanmean(per_q, axis=1) + beta * sim_crit
    df_scores["rank_score"] = -df_scores["n_no"] * 10 - df_scores["n_unclear"]

    n_missing = int(df_scores["llm_mean"].isna().sum())
    if n_missing:
        print(f"WARNING: {n_missing} records have no valid answer from any model")
    print(f"veto_pass: {int(df_scores['veto_pass'].sum())}/{len(df_scores)} records pass")

df_scores[["n_no", "n_unclear"]].value_counts().sort_index()

Loaded 694 existing scored records.


n_no  n_unclear
0     0            234
      1            158
      2             22
      3              3
      4              2
1     0             53
      1             71
      2              7
      3              5
      4              2
2     0             39
      1             49
      2              6
      4              1
3     0              8
      1              6
      2              1
      3              1
4     0              3
      1              4
      2              1
5     0             16
6     0              2
Name: count, dtype: int64

## Step 6 — Hold-out evaluation

Metrics compare the fixed decision rule and ranking-score baselines with manual labels. With the current 50-record hold-out and a 50-record dev split, results are dev metrics; no separate test set is present.

In [23]:
evaluation = {}

if len(df_hold):
    df_eval = df_hold[["record_key", "label", "split"]].merge(
        df_scores, on="record_key", how="inner"
    )
    df_eval = df_eval.dropna(subset=["llm_mean"])
    if not CONFIG["EVALUATE_TEST"]:
        df_eval = df_eval[df_eval["split"] == "dev"].reset_index(drop=True)
        print("Test split not requested; reporting dev results only.")
    DEV = (df_eval["split"] == "dev").to_numpy()
    TEST = (df_eval["split"] == "test").to_numpy()
    y = df_eval["label"].to_numpy()

    entry = {"dev": summarize_predictions(y[DEV], df_eval.loc[DEV, "veto_pass"].to_numpy())}
    if TEST.any():
        test_stats = summarize_predictions(y[TEST], df_eval.loc[TEST, "veto_pass"].to_numpy())
        test_stats["recall_ci"] = clopper_pearson(test_stats["tp"], test_stats["tp"] + test_stats["fn"])
        test_stats["precision_ci"] = clopper_pearson(test_stats["tp"], test_stats["tp"] + test_stats["fp"])
        entry["test"] = test_stats
    evaluation["veto_pass"] = entry

    for column in ("llm_mean", "llm_rerank"):
        scores = df_eval[column].to_numpy(dtype=float)
        valid = ~np.isnan(scores)
        dev_mask, test_mask = DEV & valid, TEST & valid
        if dev_mask.sum() == 0 or y[dev_mask].sum() == 0:
            continue
        threshold = threshold_for_max_f1(y[dev_mask], scores[dev_mask])
        entry = {
            "threshold": threshold,
            "dev": summarize_predictions(y[dev_mask], scores[dev_mask] >= threshold),
        }
        if test_mask.any():
            test_stats = summarize_predictions(y[test_mask], scores[test_mask] >= threshold)
            test_stats["recall_ci"] = clopper_pearson(test_stats["tp"], test_stats["tp"] + test_stats["fn"])
            test_stats["precision_ci"] = clopper_pearson(test_stats["tp"], test_stats["tp"] + test_stats["fp"])
            entry["test"] = test_stats
        evaluation[column] = entry

    table = pd.DataFrame({
        column: {
            **{f"dev_{key}": result["dev"][key] for key in ("recall", "precision", "f1", "excluded")},
            **({f"test_{key}": result["test"][key] for key in ("recall", "precision", "f1", "excluded")}
               if "test" in result else {}),
            **({"threshold": result["threshold"]} if "threshold" in result else {}),
        }
        for column, result in evaluation.items()
    }).T.sort_values("dev_f1", ascending=False)
    display(table.round(3))

    if USE_EXISTING_CORPUS_SCORES:
        effort_tag = "-".join(sorted({model.get("reasoning_effort") or "default"
                                      for model in CONFIG["MODELS"]}))
        metrics = {
            "run_on": "holdout_validation",
            "reasoning_effort": effort_tag,
            "evaluate_test": CONFIG["EVALUATE_TEST"],
            "decision_rule": "veto: 0 'no' answers and at most 1 'unclear' answer across six questions",
            "prompt_version": PROMPT_VERSION,
            "system_prompt": SYSTEM_PROMPT,
            "questions": QUESTIONS,
            "criteria_paragraph": CRITERIA_PARAGRAPH,
            "models": CONFIG["MODELS"],
            "alpha": CONFIG["ALPHA"],
            "beta": CONFIG["BETA"],
            "evaluation": json.loads(json.dumps(
                evaluation, default=lambda value: value.item() if hasattr(value, "item") else str(value)
            )),
        }
        metrics_path = os.path.join(
            CONFIG["OUTPUT_DIR"], f"04_llm_validation_metrics_holdout_{effort_tag}.json"
        )
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2, ensure_ascii=False)
        print(f"Saved hold-out metrics: {metrics_path}")
else:
    print("No manually labeled hold-out records available; evaluation skipped.")

,dev_recall,dev_precision,dev_f1,dev_excluded,test_recall,test_precision,test_f1,test_excluded,threshold
veto_pass,1.00,0.735,0.847,0.32,1.00,0.714,0.833,0.44,NaN
llm_mean,1.00,0.735,0.847,0.32,1.00,0.714,0.833,0.44,0.917
llm_rerank,0.96,0.667,0.787,0.28,0.95,0.633,0.760,0.40,0.506


Saved hold-out metrics: ../data\04_llm_validation_metrics_holdout_medium.json


In [24]:
if evaluation:
    errors = df_eval[df_eval["veto_pass"].astype(int) != df_eval["label"]]
    titles = dict(zip(df_run["record_key"], df_run["title"].fillna("")))
    notes = dict(zip(df_hold["record_key"], df_hold["notes"].fillna(""))) if "notes" in df_hold else {}
    existing_answers = (
        df_existing.set_index("record_key")["answers"].to_dict()
        if USE_EXISTING_CORPUS_SCORES else {}
    )

    for split_name, split_mask in (("dev", errors["split"] == "dev"), ("test", errors["split"] == "test")):
        if split_name == "test" and not CONFIG["EVALUATE_TEST"]:
            continue
        for _, record in errors[split_mask].iterrows():
            decision = "FALSE POSITIVE" if record["veto_pass"] else "FALSE NEGATIVE"
            title = titles.get(record["record_key"], record["record_key"])
            print(f"\n[{split_name.upper()} {decision}] {title[:105]}")
            print(f"   n_no={record['n_no']}  n_unclear={record['n_unclear']}")
            if notes.get(record["record_key"]):
                print(f"   label note: {notes[record['record_key']]}")

            if USE_EXISTING_CORPUS_SCORES:
                answers = existing_answers.get(record["record_key"], {})
                for question in QS:
                    item = answers.get(question) or {}
                    score = item.get("score")
                    score_text = f"{score:.1f}" if isinstance(score, (int, float)) else "NA"
                    evidence = (item.get("evidence") or "")[:150]
                    print(f"   {question} {score_text}  {evidence}")
            else:
                for name in MODEL_NAMES:
                    result = results[name].get(record["record_key"]) or {}
                    for question in QS:
                        answer = (result.get("answers") or {}).get(question)
                        score = ANSWER_SCORE.get(answer, np.nan)
                        evidence = ((result.get("evidence") or {}).get(question) or "")[:150]
                        prefix = f"{question} [{name}]" if len(MODEL_NAMES) > 1 else question
                        print(f"   {prefix} {score:.1f}  {evidence}")


[DEV FALSE POSITIVE] Modeling of low-risk behavior of pedestrian movement based on dynamic data analysis
   n_no=0  n_unclear=0
   label note: Q3: Crowd anomaly and security-risk detection is the central purpose, rather than monitoring mobility in a specific walking space.
   Q1 1.0  The abstract describes converting video files into data using video recognition technology and machine-learning methods.
   Q2 1.0  The study models pedestrian movement and identifies congestion and conflict risks in crowds.
   Q3 1.0  It uses real-world video to locate crowd risks and analyze pedestrian movement, congestion, and conflicts.
   Q4 1.0  The abstract states that the analysis converts real-world video files into calculable data.
   Q5 1.0  The abstract describes video files and does not restrict the data to non-visible or non-camera sensors.
   Q6 1.0  Video is converted into data to identify pedestrian states and crowd risks, rather than shown as a stimulus to participants.

[DEV FALSE POSIT

## Step 7 — Corpus exports and pseudo-labels

This step runs only when new corpus scores are generated. Existing corpus outputs are left unchanged during hold-out validation.

In [25]:
if USE_EXISTING_CORPUS_SCORES:
    print("Existing corpus outputs preserved; export and pseudo-label generation skipped.")
else:
    df_out = df_run.merge(df_scores, on="record_key", how="left")
    labels = dict(zip(df_hold["record_key"], df_hold["label"].astype(bool))) if len(df_hold) else {}
    df_out["model_decision"] = df_out["veto_pass"]
    df_out["in_holdout"] = df_out["record_key"].isin(labels)
    df_out["holdout_label"] = df_out["record_key"].map(labels)
    df_out["is_recommended"] = np.where(
        df_out["in_holdout"], df_out["holdout_label"], df_out["model_decision"]
    ).astype(bool)

    def build_answers(key):
        answers_by_question = {}
        for question in QS:
            per_model = {}
            for name in MODEL_NAMES:
                result = results[name].get(key) or {}
                answer = (result.get("answers") or {}).get(question)
                per_model[name] = {
                    "answer": answer,
                    "score": ANSWER_SCORE.get(answer),
                    "evidence": (result.get("evidence") or {}).get(question),
                }
            if len(MODEL_NAMES) == 1:
                only = next(iter(per_model.values()))
                entry = {"score": only["score"], "evidence": only["evidence"]}
            else:
                scores = [item["score"] for item in per_model.values() if item["score"] is not None]
                entry = {
                    "score": sum(scores) / len(scores) if scores else None,
                    "by_model": per_model,
                }
            entry["tier"] = (
                "yes" if entry["score"] == 1.0 else
                "no" if entry["score"] == 0.0 else
                "unclear" if entry["score"] is not None else None
            )
            answers_by_question[question] = entry
        return answers_by_question

    df_out["answers"] = df_out["record_key"].map(build_answers)
    df_out = df_out.drop(columns=[
        column for column in df_out.columns
        if column.startswith(tuple(f"{name}__" for name in MODEL_NAMES))
        or column in ("llm_text", "student_text")
    ])

    effort_tag = "-".join(sorted({model.get("reasoning_effort") or "default"
                                  for model in CONFIG["MODELS"]}))
    run_tag = f"{CONFIG['RUN_ON']}_{effort_tag}"
    scores_path = os.path.join(CONFIG["OUTPUT_DIR"], f"04_llm_scores_{run_tag}.json")
    df_out.to_json(scores_path, orient="records", indent=2, force_ascii=False)
    if CONFIG["RUN_ON"] == "corpus":
        screened_path = os.path.join(CONFIG["OUTPUT_DIR"], "04_llm_screened_corpus.json")
        df_out[df_out["is_recommended"]].to_json(
            screened_path, orient="records", indent=2, force_ascii=False
        )

        pool = df_run.merge(df_scores, on="record_key", how="left")
        pool = pool[~pool["record_key"].isin(holdout_keys) & pool["student_text"].notna()]
        pool = pool.dropna(subset=["rank_score"]).sort_values(
            ["rank_score", "llm_mean", "llm_rerank"], ascending=False
        )
        for fraction in CONFIG["PSEUDO_LABEL_FRACTIONS"]:
            assert 0 < fraction <= 0.5
            count = int(math.floor(fraction * len(pool)))
            top, bottom = pool.head(count), pool.tail(count)
            pseudo_labels = (
                [{"text": text, "label": True, "record_key": key, "llm_mean": float(score)}
                 for text, key, score in zip(top["student_text"], top["record_key"], top["llm_mean"])]
                + [{"text": text, "label": False, "record_key": key, "llm_mean": float(score)}
                   for text, key, score in zip(bottom["student_text"], bottom["record_key"], bottom["llm_mean"])]
            )
            path = os.path.join(CONFIG["PSEUDO_LABEL_DIR"], f"pseudo_llm_{int(fraction * 100)}.json")
            with open(path, "w", encoding="utf-8") as f:
                json.dump(pseudo_labels, f, indent=2, ensure_ascii=False)
            print(f"{path}: {count} positive + {count} negative")

    metrics = {
        "run_on": CONFIG["RUN_ON"],
        "reasoning_effort": effort_tag,
        "evaluate_test": CONFIG["EVALUATE_TEST"],
        "decision_rule": "veto: 0 'no' answers and at most 1 'unclear' answer across six questions",
        "prompt_version": PROMPT_VERSION,
        "system_prompt": SYSTEM_PROMPT,
        "questions": QUESTIONS,
        "criteria_paragraph": CRITERIA_PARAGRAPH,
        "models": CONFIG["MODELS"],
        "alpha": CONFIG["ALPHA"],
        "beta": CONFIG["BETA"],
        "evaluation": json.loads(json.dumps(
            evaluation, default=lambda value: value.item() if hasattr(value, "item") else str(value)
        )),
    }
    metrics_path = os.path.join(CONFIG["OUTPUT_DIR"], f"04_llm_validation_metrics_{run_tag}.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"Saved {scores_path} and validation metrics.")

Existing corpus outputs preserved; export and pseudo-label generation skipped.


## Step 8 — Optional stricter filter

Applies the stricter question-specific rule only to a newly generated corpus. It is not applied during validation-only reruns.

In [ ]:
if USE_EXISTING_CORPUS_SCORES:
    print("Strict re-filter skipped during hold-out validation.")
else:
    EXEMPT_UNCLEAR_QUESTIONS = {"Q1", "Q4"}

    def strict_pass(answers):
        if not isinstance(answers, dict):
            return False
        unclear_questions = []
        for question in QS:
            tier = (answers.get(question) or {}).get("tier")
            if tier is None or tier == "no":
                return False
            if tier == "unclear":
                unclear_questions.append(question)
        return (
            len(unclear_questions) <= 1
            and (not unclear_questions or unclear_questions[0] in EXEMPT_UNCLEAR_QUESTIONS)
        )

    df_out["strict_pass"] = df_out["answers"].map(strict_pass)
    df_out["is_recommended_strict"] = np.where(
        df_out["in_holdout"], df_out["holdout_label"], df_out["strict_pass"]
    ).astype(bool)
    before = int(df_out["is_recommended"].sum())
    after = int(df_out["is_recommended_strict"].sum())
    assert after <= before, "The stricter rule must not add records."
    print(f"Retained before: {before}; after stricter filter: {after}.")

    df_out["is_recommended"] = df_out["is_recommended_strict"]
    df_out[df_out["is_recommended"]].drop(
        columns=["strict_pass", "is_recommended_strict"]
    ).to_json(
        os.path.join(CONFIG["OUTPUT_DIR"], "04_llm_screened_corpus.json"),
        orient="records", indent=2, force_ascii=False,
    )

Strict re-filter skipped during hold-out validation.


: 